[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HumbertoDiego/AjustamentoAvancadoIME/blob/main/02_ajustamento_com_injuncoes.ipynb)

# Aula 2 - Ajustamento com Injunções

**Maj Diego - 1º Semestre / 2027**

**Objetivos:**

1. Interpretar injunções como informação adicional ou definição de datum.
2. Formular o modelo combinado com injunções.
3. Aplicar injunções ao modelo paramétrico e analisar seu efeito na solução.
4. Preparar a 1ª VE prática.

## O Problema

Considere que um primeiro ajustamento já foi concluído a partir das observações primitivas. Dele foram preservados:

- o vetor de parâmetros ajustados, $\hat X_0$;
- sua matriz de covariância, $\Sigma_{\hat X_0}$ (ou a matriz cofatora $Q_{\hat X_0}$ acompanhada do fator de variância);
- a definição do sistema de referência e das unidades.

Mais tarde, tornam-se disponíveis **novas observações**, coordenadas de controle ou relações que os parâmetros devem satisfazer. Refazer todo o ajustamento — remontando as equações normais a partir das observações primitivas — pode ser caro ou até impossível, pois os dados originais talvez não estejam mais disponíveis.

A ideia do ajustamento com injunções é aproveitar $\hat X_0$ como a melhor informação já consolidada e calcular somente a **correção** provocada pela nova informação:

$$
\hat X_{\text{novo}}=\hat X_0+\Delta X.
$$

Entretanto, não basta conhecer $\hat X_0$. A matriz $\Sigma_{\hat X_0}$ é indispensável, pois informa quais combinações de parâmetros ainda são incertas e, portanto, quanto cada uma pode ser alterada. Sem essa matriz, não é possível combinar corretamente o resultado anterior com as novas observações.

Se a nova informação puder ser escrita como

$$
L_n=A_nX+e_n,\qquad \Sigma_{L_n}=\operatorname{Cov}(e_n),
$$

e for independente das observações usadas no primeiro ajustamento, a atualização é

$$
\Delta X=
\Sigma_{\hat X_0}A_n^T
\left(A_n\Sigma_{\hat X_0}A_n^T+\Sigma_{L_n}\right)^{-1}
\left(L_n-A_n\hat X_0\right),
$$

$$
\Sigma_{\hat X_{\text{novo}}}=
\Sigma_{\hat X_0}-
\Sigma_{\hat X_0}A_n^T
\left(A_n\Sigma_{\hat X_0}A_n^T+\Sigma_{L_n}\right)^{-1}
A_n\Sigma_{\hat X_0}.
$$

O termo $L_n-A_n\hat X_0$ é a **inovação**: a discrepância entre a nova informação e o valor previsto pelo ajustamento anterior. Assim, atualizam-se a solução e sua precisão sem repetir os cálculos primitivos. Se houver correlação entre os dois conjuntos de dados, as covariâncias cruzadas precisam ser incluídas; ignorá-las leva a uma precisão excessivamente otimista.

## 1. Por que usar injunções?

Uma injunção acrescenta informação sobre os parâmetros. Ela pode cumprir funções diferentes:

1. **Definir o datum:** remover translações, rotações ou escala que não são determinadas pelas observações.
2. **Incorporar controle externo:** introduzir uma coordenada, distância, direção ou outro valor conhecido posteriormente.
3. **Representar uma relação física ou geométrica:** por exemplo, dois parâmetros iguais ou uma soma que deve permanecer constante.
4. **Atualizar um ajustamento existente:** combinar a solução anterior e sua covariância com novas informações.

Na forma linear, uma injunção é escrita como

$$
CX=W,
$$

em que cada linha de $C$ seleciona uma combinação dos parâmetros e $W$ contém o valor imposto. Antes de aplicá-la, deve-se verificar:

- compatibilidade de datum, unidades e época;
- independência linear das linhas de $C$;
- compatibilidade da injunção com as observações;
- origem e precisão de $W$.

Uma injunção **forte** é tratada como exata. Uma injunção **estocástica** reconhece que $W$ possui incerteza. Essa escolha muda tanto a estimativa dos parâmetros quanto sua matriz de covariância.

## 2. Modelo paramétrico com injunções

No modelo paramétrico linear,

$$
L+v=AX,\qquad \min(v^TPv),
$$

as equações normais do ajustamento sem injunções são

$$
N\hat X=U,\qquad N=A^TPA,\qquad U=A^TPL.
$$

Ao impor a injunção forte $CX=W$, minimiza-se $v^TPv$ sujeito a essa igualdade. Com multiplicadores de Lagrange $K$, obtém-se

$$
\begin{bmatrix}
N&C^T\\
C&0
\end{bmatrix}
\begin{bmatrix}
\hat X\\K
\end{bmatrix}
=
\begin{bmatrix}
U\\W
\end{bmatrix}.
$$

O vetor $K$ mede a reação necessária para fazer a solução respeitar a restrição. Um multiplicador elevado pode indicar conflito entre a injunção e as observações, mas sua interpretação depende das unidades e dos pesos.

Quando o ajustamento anterior já está concluído, não é necessário reconstruir $N$ e $U$. Para uma injunção forte, basta usar $\hat X_0$ e $\Sigma_{\hat X_0}$:

$$
\hat X_c=\hat X_0+
\Sigma_{\hat X_0}C^T
\left(C\Sigma_{\hat X_0}C^T\right)^{-1}
\left(W-C\hat X_0\right),
$$

$$
\Sigma_{\hat X_c}=
\Sigma_{\hat X_0}-
\Sigma_{\hat X_0}C^T
\left(C\Sigma_{\hat X_0}C^T\right)^{-1}
C\Sigma_{\hat X_0}.
$$

A primeira expressão distribui a discrepância $W-C\hat X_0$ entre os parâmetros conforme suas correlações e incertezas. A segunda mostra que a incerteza é eliminada apenas na direção restringida; as demais direções podem conservar variância.

### Exemplo numérico: solução livre, injunção forte e injunção estocástica

O exemplo abaixo ajusta dois parâmetros, preserva a solução livre e sua covariância e, em seguida, introduz o controle sobre o primeiro parâmetro. A mesma informação é aplicada de duas formas para evidenciar a diferença entre considerar o controle exato ou incerto.

In [1]:
import numpy as np

np.set_printoptions(precision=6, suppress=True)

# Ajustamento primitivo
A = np.array([[1.0, 0.0], [0.0, 1.0], [-1.0, 1.0]])
L = np.array([[100.2], [101.0], [0.9]])
P = np.eye(3)

N = A.T @ P @ A
U = A.T @ P @ L
x_livre = np.linalg.solve(N, U)
Sigma_livre = np.linalg.inv(N)  # variância de unidade de peso adotada como 1

# Nova informação: x_1 = 100
C = np.array([[1.0, 0.0]])
W = np.array([[100.0]])
inovacao = W - C @ x_livre

def atualizar(x0, Sigma0, C, W, Sigma_W=None):
    """Atualiza uma solução prévia com uma injunção forte ou estocástica."""
    if Sigma_W is None:
        Sigma_W = np.zeros((C.shape[0], C.shape[0]))
    Sigma_inovacao = C @ Sigma0 @ C.T + Sigma_W
    ganho = Sigma0 @ C.T @ np.linalg.inv(Sigma_inovacao)
    x_atualizado = x0 + ganho @ (W - C @ x0)
    Sigma_atualizada = Sigma0 - ganho @ C @ Sigma0
    return x_atualizado, Sigma_atualizada

x_forte, Sigma_forte = atualizar(x_livre, Sigma_livre, C, W)

# Controle com desvio-padrão de 0,20 unidade
Sigma_W = np.array([[0.20**2]])
x_estoc, Sigma_estoc = atualizar(x_livre, Sigma_livre, C, W, Sigma_W)

for nome, x, Sigma in [
    ("Livre", x_livre, Sigma_livre),
    ("Injunção forte", x_forte, Sigma_forte),
    ("Injunção estocástica", x_estoc, Sigma_estoc),
]:
    residuos_primitivos = A @ x - L
    print(f"{nome:22s} X = {x.ravel()}")
    print(f"{'':22s} C X - W = {(C @ x - W).item(): .6f}")
    print(f"{'':22s} resíduos = {residuos_primitivos.ravel()}")
    print(f"{'':22s} diag(Sigma_X) = {np.diag(Sigma)}\n")

# Verificações mínimas
assert np.allclose(C @ x_forte, W)
assert np.trace(Sigma_forte) <= np.trace(Sigma_estoc) <= np.trace(Sigma_livre)

Livre                  X = [100.166667 101.033333]
                       C X - W =  0.166667
                       resíduos = [-0.033333  0.033333 -0.033333]
                       diag(Sigma_X) = [0.666667 0.666667]

Injunção forte         X = [100.   100.95]
                       C X - W =  0.000000
                       resíduos = [-0.2  -0.05  0.05]
                       diag(Sigma_X) = [0.  0.5]

Injunção estocástica   X = [100.009434 100.954717]
                       C X - W =  0.009434
                       resíduos = [-0.190566 -0.045283  0.045283]
                       diag(Sigma_X) = [0.037736 0.509434]



### Como interpretar o exemplo

- A solução **livre** é o resultado do primeiro ajustamento e funciona como ponto de partida.
- A injunção **forte** faz $C\hat X=W$ exatamente. Por isso, a variância da combinação restringida torna-se nula.
- A injunção **estocástica** desloca a solução na direção do controle, mas mantém uma discrepância compatível com a incerteza atribuída a $W$.
- Os parâmetros não selecionados diretamente por $C$ também podem mudar quando estão correlacionados com o parâmetro restringido.
- Os resíduos das observações primitivas podem aumentar, pois a solução passa a conciliar essas observações com a informação adicional.

A redução da covariância não significa, por si só, que a injunção seja correta. Uma restrição incompatível também produz uma solução formalmente precisa. Por isso, devem ser analisados a inovação, os resíduos, o fator de variância e a procedência do controle.

## 3. Modelo combinado com injunções

No modelo combinado, observações e parâmetros aparecem na mesma relação funcional:

$$
F(L_a,X_a)=0.
$$

Linearizando em torno de valores aproximados $L_0$ e $X_0$, obtém-se

$$
Bv+A\Delta X+w=0,
$$

com

$$
B=\left.\frac{\partial F}{\partial L}\right|_{L_0,X_0},
\qquad
A=\left.\frac{\partial F}{\partial X}\right|_{L_0,X_0},
\qquad
w=F(L_0,X_0).
$$

As injunções são acrescentadas como equações sobre os parâmetros,

$$
G(X)=0
\quad\Longrightarrow\quad
C\Delta X+w_c=0,
$$

em que $C=\partial G/\partial X$ e $w_c=G(X_0)$. A separação entre $A$ e $B$ deve ser mantida: $A$ descreve variações dos parâmetros, enquanto $B$ descreve correções das observações.

O procedimento iterativo é:

1. escolher $L_0$ e $X_0$;
2. calcular $A$, $B$, $w$, $C$ e $w_c$;
3. resolver simultaneamente as correções $v$ e $\Delta X$;
4. atualizar $L_a=L_0+v$ e $X_a=X_0+\Delta X$;
5. repetir até que $\Delta X$ e o fechamento das equações sejam menores que as tolerâncias adotadas.

Injunções não lineares devem ser novamente linearizadas em cada iteração. Ao final, é necessário verificar tanto $F(L_a,X_a)\approx0$ quanto $G(X_a)\approx0$.

## 4. Injunção forte × estocástica

### Injunção forte

É usada quando a relação é considerada exata:

$$
CX=W.
$$

Ela elimina a variância na direção restringida e deve ser empregada com cautela. Exemplos típicos são uma definição matemática de datum ou uma identidade física assumida exata no modelo.

### Injunção estocástica

Quando o controle possui covariância $\Sigma_W$, ele é tratado como pseudo-observação:

$$
W=CX+\eta,\qquad \operatorname{Cov}(\eta)=\Sigma_W.
$$

A atualização da solução anterior torna-se

$$
\hat X_e=\hat X_0+
\Sigma_{\hat X_0}C^T
\left(C\Sigma_{\hat X_0}C^T+\Sigma_W\right)^{-1}
\left(W-C\hat X_0\right).
$$

Quanto menor $\Sigma_W$, mais a solução se aproxima da injunção forte; quanto maior $\Sigma_W$, menor é a influência do controle.

| Aspecto | Forte | Estocástica |
|---|---|---|
| Incerteza de $W$ | nula por hipótese | representada por $\Sigma_W$ |
| Atendimento a $CX=W$ | exato | aproximado e ponderado |
| Efeito na covariância | zera a variância na direção restringida | reduz, mas geralmente não zera |
| Uso recomendado | definição exata ou datum | controle medido ou informação externa incerta |

Em ambos os casos, documente origem, época, unidade, referencial e precisão da informação. Também verifique se $C\Sigma_{\hat X_0}C^T$ é invertível. Singularidade pode indicar injunções redundantes ou direções para as quais a solução anterior não fornece informação suficiente.

## 5. Verificações após a atualização

Uma solução atualizada deve ser acompanhada, no mínimo, das seguintes verificações:

1. **Fechamento da injunção:** avaliar $C\hat X-W$.
2. **Magnitude da inovação:** comparar $W-C\hat X_0$ com sua covariância
   $C\Sigma_{\hat X_0}C^T+\Sigma_W$.
3. **Resíduos:** verificar se a nova informação produz resíduos anormalmente grandes nas observações primitivas.
4. **Precisão:** comparar $\Sigma_{\hat X_0}$ e $\Sigma_{\hat X_{\text{novo}}}$, observando quais direções realmente ganharam informação.
5. **Posto e condicionamento:** identificar injunções redundantes, dependências lineares e sistemas numericamente instáveis.
6. **Consistência do modelo:** confirmar datum, unidades, época e eventuais correlações entre dados antigos e novos.

Para uma única inovação, o valor

$$
T=(W-C\hat X_0)^T
\left(C\Sigma_{\hat X_0}C^T+\Sigma_W\right)^{-1}
(W-C\hat X_0)
$$

é uma medida adimensional de incompatibilidade. Valores elevados sugerem que a nova informação, sua precisão ou o ajustamento anterior devem ser investigados antes da atualização definitiva.

## 1ª VE prática

**Tarefa:** ajustar uma pequena rede inicialmente sem injunções. Em seguida, conservar apenas $\hat X_0$ e $\Sigma_{\hat X_0}$ e introduzir uma nova informação de controle sem remontar as equações normais primitivas.

**Etapas mínimas:**

1. apresentar o modelo funcional e estocástico do ajustamento inicial;
2. calcular a solução livre e sua matriz de covariância;
3. formular uma injunção forte e uma versão estocástica da mesma informação;
4. atualizar a solução a partir de $\hat X_0$ e $\Sigma_{\hat X_0}$;
5. comparar parâmetros, resíduos, covariâncias, inovação e condicionamento;
6. discutir se a nova informação é compatível com o ajustamento anterior.

**Entregáveis:** notebook executável, formulação matricial, resultados comentados, verificações numéricas e conclusão técnica sobre o efeito de cada tipo de injunção.